In [ ]:
# Instalaivimas reikalingų paketų
#!pip install git+https://github.com/huggingface/transformers accelerate
!pip install sentencepiece
print("Instaliavimas baigtas")

In [ ]:
# Perspėjimų ignoravimas
import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
from PIL import Image
import requests
import torch
import os
import re
import pandas as pd
print("Importavimas baigtas")

In [ ]:
# Modelių naudojimas
processor = LlavaNextProcessor.from_pretrained("llava-hf/llava-v1.6-vicuna-13b-hf")
model = LlavaNextForConditionalGeneration.from_pretrained("llava-hf/llava-v1.6-vicuna-13b-hf", torch_dtype=torch.float16, low_cpu_mem_usage=True) 
model.to("cuda:0")

#processor = LlavaNextProcessor.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf")
#model = LlavaNextForConditionalGeneration.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf", torch_dtype=torch.float16, low_cpu_mem_usage=True) 
#model.to("cuda:0")

print("Modelio krovimas baigtas")

In [ ]:
# Lentelės užkrovimas, generavimas ir saugojimas
df = pd.read_csv('flicker8k_edited.csv', sep=';')
import numpy as np

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)


# Išsivalome lentelę
columns_to_clear = [
    'Generated Caption', 'BLEU', 'ROGUE', 'METEOR',
    'BERTscore Precision', 'BERTscore Recall', 'BERTscore F1', 'SentenceBERT'
]
for col in columns_to_clear:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: '')

image_folder = "Flickr8k"

#for index, row in df.head(20).iterrows():
for index, row in df.iterrows():
    filename = row["Image Name"]
    image_path = os.path.join(image_folder, filename)
    try:
        image = Image.open(image_path)
        conversation = [
            {
              "role": "user",
              "content": [
            {"type": "text", "text": "Keep it short. No more than 10 words. The text is the caption for the image. Do not write any additional text."},
              {"type": "image"},
                ],
            },
        ]
        prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

        inputs = processor(images=image, text=prompt, return_tensors="pt").to("cuda:0")

    # autoregressively complete prompt
        

        output = model.generate(
            **inputs,
            max_new_tokens=100,
            pad_token_id=model.config.eos_token_id
        )

        generated_caption = (processor.decode(output[0], skip_special_tokens=True))
        generated_caption = str(generated_caption)
        
    # Remove everything between "INST" and "leave"
        #generated_caption = re.sub(r'\[INST\].*?\[/INST\]', '', generated_caption, flags=re.DOTALL).strip()
        #generated_caption = generated_caption.strip('"')

    # Remove everything between "INST" and "leave"
        generated_caption = re.sub(r'USER:.*?ASSISTANT:', '', generated_caption, flags=re.DOTALL).strip()
        generated_caption = generated_caption.strip('"')
        
        # Įrašymas į failą
        df.at[index, "Generated Caption"] = generated_caption

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        df.at[index, "Generated Caption"] = f"Error: {e}"

# Saugojimas
df.to_csv("Results/flicker8k_en_11.csv", sep=';', index=False)
print("Generavimas baigtas.")